# RAG Agent — Grocery Product Q&A

Same product catalog as the vector search notebook. Instead of just returning matches, we retrieve relevant products and pass them to GPT-4o to answer questions grounded in real data.

In [1]:
import os, json, time
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536

def embed(texts: list[str]) -> list[list[float]]:
    res = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in res.data]

In [2]:
# --- Product catalog: eggs, milk, and bread ---

products = [
    # === EGGS ===
    {
        "id": "egg-001",
        "category": "eggs",
        "name": "Organic Free-Range Large Brown Eggs 12ct",
        "text": (
            "Organic free-range large brown eggs from pasture-raised hens fed a "
            "certified organic, non-GMO diet. Each hen has at least 108 sqft of "
            "outdoor access. Rich golden yolks with firm whites, excellent for "
            "baking and breakfast. USDA Organic certified. 12 count carton. "
            "70 calories per egg, 6g protein, 5g fat. Refrigerate at 35-40°F. "
            "Sourced from small family farms in the Pacific Northwest."
        ),
        "price": 6.99,
        "organic": True,
    },
    {
        "id": "egg-002",
        "category": "eggs",
        "name": "Cage-Free Large White Eggs 18ct",
        "text": (
            "Cage-free large white eggs from hens housed in open barns with "
            "freedom to roam, perch, and nest. Consistent size for reliable "
            "cooking results. Great all-purpose egg for scrambling, frying, and "
            "hard boiling. 18 count value pack. 70 calories per egg, 6g protein. "
            "American Humane Certified. Best used within 3 weeks of purchase."
        ),
        "price": 4.49,
        "organic": False,
    },
    {
        "id": "egg-003",
        "category": "eggs",
        "name": "Omega-3 Enriched Eggs 12ct",
        "text": (
            "Omega-3 enriched eggs from hens fed a diet supplemented with "
            "flaxseed and fish oil. Each egg contains 225mg of Omega-3 fatty "
            "acids — three times more than conventional eggs. Supports heart "
            "and brain health. Large brown eggs, 12 count. Ideal for health-"
            "conscious consumers looking for added nutritional benefits without "
            "supplements. Vegetarian-fed hens, antibiotic-free."
        ),
        "price": 5.29,
        "organic": False,
    },
    {
        "id": "egg-004",
        "category": "eggs",
        "name": "Liquid Egg Whites 32oz Carton",
        "text": (
            "100% liquid egg whites, pasteurized and ready to pour. Fat-free, "
            "cholesterol-free, with 5g protein per serving. Perfect for omelets, "
            "smoothies, protein shakes, and baking. No cracking or separating "
            "needed. 32oz resealable carton equals approximately 20 egg whites. "
            "Popular with athletes and fitness enthusiasts for lean protein. "
            "Refrigerate after opening; use within 7 days."
        ),
        "price": 4.99,
        "organic": False,
    },

    # === MILK ===
    {
        "id": "milk-001",
        "category": "milk",
        "name": "Organic Whole Milk 1 Gallon",
        "text": (
            "USDA Organic whole milk from grass-fed cows, ultra-pasteurized for "
            "extended freshness. Rich, creamy taste with 3.25% milkfat. Excellent "
            "source of calcium, vitamin D, and protein. No antibiotics, synthetic "
            "hormones, or pesticides. Each 8oz serving has 150 calories, 8g fat, "
            "8g protein. Homogenized. Sourced from certified organic dairy farms "
            "in Vermont. Gallon jug with twist-off cap."
        ),
        "price": 7.49,
        "organic": True,
    },
    {
        "id": "milk-002",
        "category": "milk",
        "name": "2% Reduced Fat Milk Half Gallon",
        "text": (
            "2% reduced fat milk, pasteurized and homogenized. A lighter option "
            "that retains the creamy taste of whole milk with less fat. 120 "
            "calories per 8oz serving, 5g fat, 8g protein. Fortified with "
            "vitamins A and D. Versatile for drinking, cereal, coffee, and "
            "cooking. Half gallon container. Grade A. Conventional dairy, "
            "rBST-free. Keep refrigerated at 33-40°F."
        ),
        "price": 3.29,
        "organic": False,
    },
    {
        "id": "milk-003",
        "category": "milk",
        "name": "Oat Milk Original 64oz",
        "text": (
            "Plant-based oat milk made from gluten-free Swedish oats. Creamy, "
            "smooth texture that froths well for lattes and cappuccinos. Dairy-"
            "free, nut-free, soy-free. Fortified with calcium, vitamin D, and "
            "B12. 120 calories per 8oz serving, 5g fat, 3g protein. No added "
            "sugar in the original variety. Shelf-stable until opened, then "
            "refrigerate and use within 10 days. Popular vegan alternative."
        ),
        "price": 5.49,
        "organic": False,
    },
    {
        "id": "milk-004",
        "category": "milk",
        "name": "Chocolate Whole Milk 16oz Single",
        "text": (
            "Rich chocolate whole milk made with real cocoa and cane sugar. A "
            "delicious post-workout recovery drink with an ideal 4:1 carb-to-"
            "protein ratio. 280 calories per bottle, 8g fat, 8g protein, 38g "
            "sugar. Single-serve 16oz bottle, grab-and-go convenience. Great for "
            "kids' lunchboxes or as an afternoon treat. rBST-free, Grade A milk. "
            "Keep refrigerated."
        ),
        "price": 2.49,
        "organic": False,
    },

    # === BREAD ===
    {
        "id": "bread-001",
        "category": "bread",
        "name": "Organic Sourdough Loaf",
        "text": (
            "Artisan sourdough bread made with organic flour, water, salt, and "
            "a 50-year-old wild yeast starter. Slow-fermented for 24 hours, "
            "producing a tangy flavor, chewy crumb, and crispy crust. Naturally "
            "leavened — no commercial yeast or preservatives. Easier to digest "
            "than conventional bread due to long fermentation. 120 calories per "
            "slice, 4g protein. Baked fresh daily in small batches."
        ),
        "price": 6.49,
        "organic": True,
    },
    {
        "id": "bread-002",
        "category": "bread",
        "name": "Whole Wheat Sandwich Bread 20oz",
        "text": (
            "100% whole wheat sandwich bread with no high-fructose corn syrup. "
            "Soft texture with a mild nutty flavor, perfect for sandwiches, "
            "toast, and French toast. Made with whole grain wheat flour as the "
            "first ingredient. 110 calories per slice, 4g protein, 3g fiber. "
            "Contains 16g whole grains per slice. Pre-sliced, 20oz loaf. "
            "Stays fresh up to 10 days. Heart-healthy choice."
        ),
        "price": 3.99,
        "organic": False,
    },
    {
        "id": "bread-003",
        "category": "bread",
        "name": "Gluten-Free Multigrain Bread 15oz",
        "text": (
            "Gluten-free multigrain bread made with brown rice flour, tapioca "
            "starch, millet, and quinoa. Certified gluten-free for celiac and "
            "gluten-sensitive consumers. Soft texture without the grittiness "
            "common in gluten-free breads. 100 calories per slice, 2g protein, "
            "2g fiber. Dairy-free, soy-free, non-GMO. Pre-sliced 15oz loaf. "
            "Best stored in the freezer; toast from frozen for best results."
        ),
        "price": 6.99,
        "organic": False,
    },
    {
        "id": "bread-004",
        "category": "bread",
        "name": "Brioche Burger Buns 8ct",
        "text": (
            "Rich, buttery brioche burger buns made with European-style butter "
            "and cage-free eggs. Golden crust with a pillowy-soft interior that "
            "holds up to juicy burgers without falling apart. Lightly sweetened "
            "with honey. 210 calories per bun. 8 count package. Perfect for "
            "backyard grilling, pulled pork sandwiches, or breakfast sandwiches. "
            "Restaurant-quality buns at home."
        ),
        "price": 4.99,
        "organic": False,
    },
]

In [3]:
# --- Create index and upsert all products ---

INDEX = "grocery-products"

if INDEX in [i.name for i in pc.list_indexes()]:
    pc.delete_index(INDEX)

pc.create_index(
    name=INDEX,
    dimension=EMBED_DIM,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
time.sleep(1)
index = pc.Index(INDEX)

texts = [p["text"] for p in products]
embeddings = embed(texts)

vectors = [
    {
        "id": p["id"],
        "values": emb,
        "metadata": {
            "name": p["name"],
            "category": p["category"],
            "price": p["price"],
            "organic": p["organic"],
            "text": p["text"],
        },
    }
    for p, emb in zip(products, embeddings)
]

index.upsert(vectors=vectors)
time.sleep(2)

print(f"Indexed {index.describe_index_stats().total_vector_count} products")

/Users/charlierobison/Desktop/Github/charlie-robison/hfa-ai-training/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed 12 products


In [4]:
# --- RAG: retrieve context, then generate an answer ---

def ask(question: str, top_k: int = 4) -> str:
    """Retrieve relevant products, pass them to GPT-4o, return the answer."""

    # 1. Retrieve — embed the question and search Pinecone
    vec = embed([question])[0]
    results = index.query(vector=vec, top_k=top_k, include_metadata=True)

    # 2. Augment — build the prompt with retrieved product data
    context = "\n\n".join(
        f"[{m.metadata['name']} | ${m.metadata['price']:.2f} | {m.metadata['category']}]\n"
        f"{m.metadata['text']}"
        for m in results.matches
    )

    # 3. Generate — GPT-4o answers using ONLY the retrieved context
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful grocery store assistant. Answer the "
                    "customer's question using ONLY the product information "
                    "provided below. Include specific details like prices, "
                    "nutrition facts, and product names. If the information "
                    "isn't in the context, say so."
                ),
            },
            {
                "role": "user",
                "content": f"PRODUCTS:\n{context}\n\nQUESTION: {question}",
            },
        ],
    )

    answer = response.choices[0].message.content

    # Print what was retrieved and the answer
    print(f"Q: {question}")
    print(f"   Retrieved: {', '.join(m.metadata['name'] for m in results.matches)}")
    print(f"\nA: {answer}\n")
    print("-" * 60)
    return answer

In [5]:
# --- Ask questions grounded in the product catalog ---

ask("I'm training for a marathon and need high-protein, low-fat options. What do you recommend?")

ask("My kid has a gluten allergy and is lactose intolerant. What can they eat for breakfast?")

ask("I want to make a nice brunch. What eggs and bread would pair well together?")

ask("What's the cheapest option in each category?")

ask("Compare the organic products — are they worth the extra cost?")

Q: I'm training for a marathon and need high-protein, low-fat options. What do you recommend?
   Retrieved: Liquid Egg Whites 32oz Carton, 2% Reduced Fat Milk Half Gallon, Whole Wheat Sandwich Bread 20oz, Chocolate Whole Milk 16oz Single

A: For high-protein, low-fat options, I recommend the Liquid Egg Whites 32oz Carton. It is fat-free and cholesterol-free, providing 5g of protein per serving. It's perfect for omelets, smoothies, protein shakes, and baking, which can fit well into your marathon training diet. The carton is priced at $4.99 and contains approximately 20 egg whites. Remember to refrigerate it after opening and use it within 7 days.

------------------------------------------------------------
Q: My kid has a gluten allergy and is lactose intolerant. What can they eat for breakfast?
   Retrieved: Gluten-Free Multigrain Bread 15oz, Oat Milk Original 64oz, Whole Wheat Sandwich Bread 20oz, Chocolate Whole Milk 16oz Single

A: For breakfast, your kid with a gluten allergy and

'The organic products listed — Organic Whole Milk, Organic Free-Range Large Brown Eggs, and Organic Sourdough Loaf — each offer specific benefits that may justify their higher cost compared to conventional alternatives:\n\n1. **Organic Whole Milk 1 Gallon ($7.49):**\n   - **Benefits:** It is USDA Organic and sourced from grass-fed cows, free from antibiotics, synthetic hormones, or pesticides.\n   - **Nutritional Value:** Provides an excellent source of calcium, vitamin D, and protein, with each 8oz serving containing 150 calories and 8g of protein.\n   - **Additional Features:** Ultra-pasteurized for extended freshness and sourced from certified organic dairy farms, enhancing both quality and sustainability.\n\n2. **Organic Free-Range Large Brown Eggs 12ct ($6.99):**\n   - **Benefits:** These eggs come from hens with a certified organic, non-GMO diet and ample outdoor access (at least 108 sqft per hen), which may contribute to better animal welfare and egg quality.\n   - **Nutritional

In [6]:
# --- Cleanup ---
pc.delete_index(INDEX)
print(f"Deleted index '{INDEX}'")

Deleted index 'grocery-products'
